In [2]:
import pandas as pd
from Bio import AlignIO

In [2]:

# 1-letter → 3-letter conversion
aa1to3 = {
    'A': 'ALA', 'R': 'ARG', 'N': 'ASN', 'D': 'ASP',
    'C': 'CYS', 'Q': 'GLN', 'E': 'GLU', 'G': 'GLY',
    'H': 'HIS', 'I': 'ILE', 'L': 'LEU', 'K': 'LYS',
    'M': 'MET', 'F': 'PHE', 'P': 'PRO', 'S': 'SER',
    'T': 'THR', 'W': 'TRP', 'Y': 'TYR', 'V': 'VAL',
    '-': '-'  # gaps
}

In [4]:

# --- Load MSA ---
msa_file = "../cluster_0_1_mixed_MSTA_aa_plus_ec6098ref_2.faa"
msa = AlignIO.read(msa_file, "fasta")

ref_seq_id = "EC6098_reference"  # replace with your reference ID


In [5]:
# --- Build reference position -> MSA column mapping ---
def build_ref_map(msa, ref_seq_id):
    ref_record = next(r for r in msa if r.id == ref_seq_id)
    ref_map = {}
    ref_pos = 1  # 0-based
    for col_idx, aa in enumerate(ref_record.seq):
        if aa != '-':
            ref_map[ref_pos] = col_idx
            ref_pos += 1
    return ref_map

ref_map = build_ref_map(msa, ref_seq_id)

In [ ]:
ref_map

{1: 0,
 2: 1,
 3: 2,
 4: 3,
 5: 4,
 6: 5,
 7: 6,
 8: 7,
 9: 8,
 10: 9,
 11: 10,
 12: 11,
 13: 12,
 14: 13,
 15: 14,
 16: 15,
 17: 16,
 18: 17,
 19: 18,
 20: 19,
 21: 20,
 22: 21,
 23: 22,
 24: 23,
 25: 24,
 26: 25,
 27: 26,
 28: 27,
 29: 28,
 30: 29,
 31: 30,
 32: 31,
 33: 32,
 34: 33,
 35: 34,
 36: 35,
 37: 36,
 38: 37,
 39: 38,
 40: 39,
 41: 40,
 42: 41,
 43: 42,
 44: 43,
 45: 44,
 46: 45,
 47: 46,
 48: 47,
 49: 48,
 50: 49,
 51: 50,
 52: 51,
 53: 52,
 54: 53,
 55: 54,
 56: 55,
 57: 56,
 58: 57,
 59: 58,
 60: 59,
 61: 60,
 62: 61,
 63: 62,
 64: 63,
 65: 64,
 66: 65,
 67: 66,
 68: 67,
 69: 68,
 70: 69,
 71: 70,
 72: 71,
 73: 72,
 74: 73,
 75: 74,
 76: 75,
 77: 76,
 78: 77,
 79: 78,
 80: 79,
 81: 80,
 82: 81,
 83: 82,
 84: 83,
 85: 84,
 86: 85,
 87: 86,
 88: 87,
 89: 88,
 90: 89,
 91: 90,
 92: 91,
 93: 92,
 94: 93,
 95: 94,
 96: 95,
 97: 96,
 98: 97,
 99: 98,
 100: 99,
 101: 100,
 102: 101,
 103: 102,
 104: 103,
 105: 104,
 106: 105,
 107: 106,
 108: 107,
 109: 108,
 110: 109,
 111: 11

In [9]:
df_surfaces = pd.read_csv("surface_area_A53.txt", skiprows=[0,1,2],sep="\t|:", names=["chain","position", "surface_area"])
df_surfaces
df_surfaces["msa_position"] = df_surfaces["position"].map(ref_map)
df_surfaces

/var/folders/6c/dkl_rmf913l9h4s3pys_spm80000gn/T/ipykernel_48595/1730422724.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_surfaces = pd.read_csv("surface_area_A53.txt", skiprows=[0,1,2],sep="\t|:", names=["chain","position", "surface_area"])


,chain,position,surface_area,msa_position
0,#2//chain_id='A-53',4,117.855551,3
1,#2//chain_id='A-53',5,25.047236,4
2,#2//chain_id='A-53',6,209.909256,5
3,#2//chain_id='A-53',7,126.027098,6
4,#2//chain_id='A-53',8,6.739850,7
...,...,...,...,...
470,#2//chain_id='A-53',562,47.255225,561
471,#2//chain_id='A-53',563,5.316433,562
472,#2//chain_id='A-53',564,3.303428,563
473,#2//chain_id='A-53',565,27.835667,564


In [181]:
def build_msa_to_prot_pos(seq):
    msa_to_pos = [None] * len(seq)
    pos = 1
    for i, aa in enumerate(seq):
        if aa != "-":
            msa_to_pos[i] = pos
            pos += 1

    return msa_to_pos

In [11]:
expanded_rows = []

A_cols = df_surfaces["msa_position"].to_numpy()

for record in msa:
    # if record.id == ref_seq_id:
    #     continue
    target_seq = record.seq
    msa_to_prot_pos = build_msa_to_prot_pos(target_seq)

    for i, row in df_surfaces.iterrows():
        new_row = row.copy()

        A_col = A_cols[i]

        if pd.notna(A_col):
            A_col = int(A_col)
            aa = target_seq[A_col]
            new_row["A_AA"] = aa1to3.get(aa, "-") if aa != "-" else "-"
            new_row["A_pos"] = msa_to_prot_pos[A_col] or "-"
        else:
            new_row["A_AA"] = "-"
            new_row["A_pos"] = "-"

        new_row["protein"] = record.id
        expanded_rows.append(new_row)


In [12]:
df_expanded = pd.DataFrame(expanded_rows)

In [13]:
df_expanded

,chain,position,surface_area,msa_position,A_AA,A_pos,protein
0,#2//chain_id='A-53',4,117.855551,3,PHE,4,EC6098_reference
1,#2//chain_id='A-53',5,25.047236,4,GLY,5,EC6098_reference
2,#2//chain_id='A-53',6,209.909256,5,ARG,6,EC6098_reference
3,#2//chain_id='A-53',7,126.027098,6,LYS,7,EC6098_reference
4,#2//chain_id='A-53',8,6.739850,7,VAL,8,EC6098_reference
...,...,...,...,...,...,...,...
470,#2//chain_id='A-53',562,47.255225,561,PHE,511,MGYP003333944615
471,#2//chain_id='A-53',563,5.316433,562,-,-,MGYP003333944615
472,#2//chain_id='A-53',564,3.303428,563,-,-,MGYP003333944615
473,#2//chain_id='A-53',565,27.835667,564,-,-,MGYP003333944615


In [15]:
df_expanded.loc[df_expanded['protein'].str.contains("ref")]

,chain,position,surface_area,msa_position,A_AA,A_pos,protein
0,#2//chain_id='A-53',4,117.855551,3,PHE,4,EC6098_reference
1,#2//chain_id='A-53',5,25.047236,4,GLY,5,EC6098_reference
2,#2//chain_id='A-53',6,209.909256,5,ARG,6,EC6098_reference
3,#2//chain_id='A-53',7,126.027098,6,LYS,7,EC6098_reference
4,#2//chain_id='A-53',8,6.739850,7,VAL,8,EC6098_reference
...,...,...,...,...,...,...,...
470,#2//chain_id='A-53',562,47.255225,561,LEU,562,EC6098_reference
471,#2//chain_id='A-53',563,5.316433,562,ILE,563,EC6098_reference
472,#2//chain_id='A-53',564,3.303428,563,ASP,564,EC6098_reference
473,#2//chain_id='A-53',565,27.835667,564,HIS,565,EC6098_reference


In [14]:
df_expanded.to_csv("mapped_positions.tsv", sep='\t', index=False)


In [3]:
df_expanded = pd.read_csv("mapped_positions.tsv", sep='\t')
df_expanded
df_expanded["A_pos"] = pd.to_numeric(df_expanded["A_pos"], errors="coerce").astype("Int64")


In [ ]:
# df_feature_imp = pd.read_csv("/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/RF_models_automation/RF_results/Rep_Seq_Position_Mapping/example_cluster_0_1_mixed/new_model/cluster_0_1_mixed_MSTA_aa_plus_ec6098re_EC6098_reference_feature_importance.tsv",
#                              sep='\t',usecols=["index","gini_imp","msa_position"])
# df_feature_imp
# df_feature_imp = df_feature_imp.sort_values('gini_imp', ascending=False).drop_duplicates('msa_position',keep='first').reset_index().reset_index(names="feat_rank")
# df_feature_imp

# # NOTE: Because it is only based on positions and not pos+AA, we sort by the highest scoring pos+AA for each position (e.g. between 80_S and 80_P, choose highest ranking) and drop duplicates

# df_feature_imp = df_feature_imp.sort_values('gini_imp', ascending=False)

# # df_feature_imp = df_feature_imp.loc[df_feature_imp['feat_rank'] < 1000]
# df_feature_imp


,feat_rank,level_0,index,gini_imp,msa_position
0,0,0,435_W,0.010456,435
1,1,1,330_K,0.009997,330
2,2,2,80_P,0.009928,80
3,3,4,324_Q,0.009069,324
4,4,5,445_F,0.008860,445
...,...,...,...,...,...
561,561,2477,203_P,0.000054,203
562,562,2641,55_P,0.000050,55
563,563,2656,412_H,0.000049,412
564,564,2695,442_E,0.000048,442


In [4]:
df_feature_imp = pd.read_csv("/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/Microviridae_analysis/multiple_cluster_analysis/combined_model_phyloglm/phyloglm_intersect_giniscores_top200.tsv",
                             sep='\t')
print(df_feature_imp.shape)

# 
df_feature_imp[['pos','AA']] = df_feature_imp['feature'].str.split("_",expand=True)
df_feature_imp['pos'] = df_feature_imp['pos'].astype('int64')
df_feature_imp
df_feature_imp = df_feature_imp.sort_values('feature_importance_vals', ascending=False).drop_duplicates('pos',keep='first')
df_feature_imp

(200, 13)


,Unnamed: 0,feature,Estimate,SE,z.value,p.value,alpha,padj,imp_order,feature_importance_vals,abs_z.value,abs_Estimate,biome_predictor,pos,AA
0,0,435_W,-3.139176,0.494326,-6.350420,2.147274e-10,0.559351,1.663444e-08,0,0.010456,6.350420,3.139176,lake,435,W
1,1,330_K,-2.588086,0.476886,-5.427056,5.729117e-08,0.521602,2.219109e-06,1,0.009997,5.427056,2.588086,lake,330,K
2,2,80_S,4.451964,1.220548,3.647513,2.647912e-04,0.230644,2.729167e-03,3,0.009723,3.647513,4.451964,ocean,80,S
3,3,324_Q,-2.301889,0.501361,-4.591284,4.405279e-06,0.576690,1.002775e-04,4,0.009069,4.591284,2.301889,lake,324,Q
4,4,445_F,-2.323584,0.722366,-3.216630,1.297059e-03,0.468187,9.599039e-03,5,0.008860,3.216630,2.323584,lake,445,F
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191,191,61_I,3.818933,0.846828,4.509688,6.492294e-06,0.257909,1.398318e-04,245,0.000658,4.509688,3.818933,ocean,61,I
192,192,48_I,1.440823,0.244429,5.894636,3.755082e-09,0.304718,1.939318e-07,247,0.000655,5.894636,1.440823,ocean,48,I
194,194,353_T,1.287417,0.330417,3.896343,9.765593e-05,0.294043,1.215133e-03,251,0.000650,3.896343,1.287417,ocean,353,T
195,195,63_A,1.368494,0.404437,3.383699,7.151647e-04,0.316119,6.047423e-03,252,0.000646,3.383699,1.368494,ocean,63,A


In [5]:
#merge with duplicated(all) interactions
# Merge A features (keep all)
merged_A = df_expanded.merge(df_feature_imp
    , how="left", left_on="A_pos", right_on="pos")
# merged_A.sort_values('feat_rank')

In [7]:
import numpy as np
df_expanded = merged_A.loc[~((merged_A["A_AA"] == '-') | (merged_A["A_AA"] == None))]
df_expanded

,chain,position,surface_area,msa_position,A_AA,A_pos,protein,Unnamed: 0,feature,Estimate,...,p.value,alpha,padj,imp_order,feature_importance_vals,abs_z.value,abs_Estimate,biome_predictor,pos,AA
0,#2//chain_id='A-53',4,117.855551,3,PHE,4,EC6098_reference,148.0,4_G,2.525729,...,0.000040,0.299638,0.000575,178.0,0.000896,4.108864,2.525729,ocean,4.0,G
1,#2//chain_id='A-53',5,25.047236,4,GLY,5,EC6098_reference,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,#2//chain_id='A-53',6,209.909256,5,ARG,6,EC6098_reference,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,#2//chain_id='A-53',7,126.027098,6,LYS,7,EC6098_reference,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,#2//chain_id='A-53',8,6.739850,7,VAL,8,EC6098_reference,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527716,#2//chain_id='A-53',558,1.568663,557,GLY,507,MGYP003333944615,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
527717,#2//chain_id='A-53',559,30.396764,558,THR,508,MGYP003333944615,173.0,508_R,2.785906,...,0.000168,0.256933,0.001894,221.0,0.000719,3.762798,2.785906,ocean,508.0,R
527718,#2//chain_id='A-53',560,3.078450,559,PRO,509,MGYP003333944615,57.0,509_D,2.340584,...,0.000007,0.302482,0.000146,65.0,0.002693,4.494466,2.340584,ocean,509.0,D
527719,#2//chain_id='A-53',561,2.415762,560,THR,510,MGYP003333944615,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
data_df = pd.read_csv("/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/RF_models_automation/RF_results/raw/cluster_0_1_mixed_MSTA_aa_plus_ec6098re_data_df.tsv", sep='\t',usecols=['protein','ecosystem_subtype'])
data_df

,protein,ecosystem_subtype
0,IMGVR_UViG_3300027969_000035|3300027969|Ga0209...,Lake
1,IMGVR_UViG_3300027969_000036|3300027969|Ga0209...,Lake
2,IMGVR_UViG_3300027973_000026|3300027973|Ga0209...,Lake
3,IMGVR_UViG_3300027974_000782|3300027974|Ga0209...,Lake
4,IMGVR_UViG_3300028553_000004|3300028553|Ga0247...,Lake
...,...,...
1246,MGYP001269301632,Oceanic
1247,MGYP001445468529,Oceanic
1248,MGYP000515316755,Lake
1249,MGYP003325855982,Oceanic


In [9]:
df_expand_biomes = df_expanded.set_index('protein').join(data_df.set_index('protein'),how='left')

In [10]:
df_expand_biomes

,chain,position,surface_area,msa_position,A_AA,A_pos,Unnamed: 0,feature,Estimate,SE,...,alpha,padj,imp_order,feature_importance_vals,abs_z.value,abs_Estimate,biome_predictor,pos,AA,ecosystem_subtype
protein,,,,,,,,,,,,,,,,,,,,,
EC6098_reference,#2//chain_id='A-53',4,117.855551,3,PHE,4,148.0,4_G,2.525729,0.614702,...,0.299638,0.000575,178.0,0.000896,4.108864,2.525729,ocean,4.0,G,NaN
EC6098_reference,#2//chain_id='A-53',5,25.047236,4,GLY,5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EC6098_reference,#2//chain_id='A-53',6,209.909256,5,ARG,6,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EC6098_reference,#2//chain_id='A-53',7,126.027098,6,LYS,7,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EC6098_reference,#2//chain_id='A-53',8,6.739850,7,VAL,8,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MGYP003333944615,#2//chain_id='A-53',558,1.568663,557,GLY,507,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Oceanic
MGYP003333944615,#2//chain_id='A-53',559,30.396764,558,THR,508,173.0,508_R,2.785906,0.740381,...,0.256933,0.001894,221.0,0.000719,3.762798,2.785906,ocean,508.0,R,Oceanic
MGYP003333944615,#2//chain_id='A-53',560,3.078450,559,PRO,509,57.0,509_D,2.340584,0.520770,...,0.302482,0.000146,65.0,0.002693,4.494466,2.340584,ocean,509.0,D,Oceanic


In [11]:
df_expand_biomes.reset_index().to_csv("mapped_positions_with_biomes_2.tsv", sep='\t', index=False)